# अध्याय 7: चॅट अनुप्रमाणे तयार करणे
## Microsoft Foundry मॉडेल्स API जलद प्रारंभ

हा नोटबुक [Azure OpenAI नमुने संग्रहालय](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) मधून रूपांतरित केला आहे ज्यामध्ये [Azure OpenAI](notebook-azure-openai.ipynb) सेवा वापरणारे नोटबुक्स आहेत.

> **टीप:** GitHub मॉडेल्स जुलै 2026 च्या अखेरीस बंद होत आहे. हा नोटबुक आता [Microsoft Foundry मॉडेल्स](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) कडे लक्ष केंद्रित करतो, जे समान विनामूल्य चाचणीसाठी मॉडेल कॅटलॉग आणि Azure AI Inference SDK अनुभव पुरवते.


# विहंगावलोकन  
"मोठे भाषा मॉडेल म्हणजे फंक्शन्स जे मजकूराला मजकूरात रूपांतरित करतात. इनपुट मजकूर दिल्यास, मोठे भाषा मॉडेल पुढील येणारा मजकूर काय असेल हे भाकीत करण्याचा प्रयत्न करते"(1). हा "क्विकस्टार्ट" नोटबुक वापरकर्त्यांना उच्च-स्तरीय LLM संकल्पना, AML सह सुरूवात करण्यासाठी आवश्यक कोअर पॅकेजेस, प्रॉम्प्ट डिझाइनची मऊ ओळख, आणि विविध वापर प्रकरणांच्या काही लहान उदाहरणांशी परिचित करून देईल. 


## Table of Contents  

[Overview](#overview)  
[How to use OpenAI Service](#how-to-use-openai-service)  
[1. Creating your OpenAI Service](#1.-creating-your-openai-service)  
[2. Installation](#2.-installation)    
[3. Credentials](#3.-credentials)  

[Use Cases](#use-cases)    
[1. Summarize Text](#1.-summarize-text)  
[2. Classify Text](#2.-classify-text)  
[3. Generate New Product Names](#3.-generate-new-product-names)  
[4. Fine Tune a Classifier](#4.fine-tune-a-classifier)  

[References](#references)


### आपला पहिला प्रॉम्प्ट तयार करा  
हा लहान व्यायाम Microsoft Foundry Models मध्ये "सारांश" या सोप्या कामासाठी मॉडेलला प्रॉम्प्ट सादर करण्यासाठी मूलभूत ओळख देईल.  


**पायऱ्या**:  
1. आपल्याकडे नसेल तर आपल्या पायथन वातावरणात `azure-ai-inference` लायब्ररी इन्स्टॉल करा.  
2. मानक सहाय्यक लायब्ररीज लोड करा आणि आपले Microsoft Foundry Models प्रमाणपत्र सेट करा.  
3. आपल्या कामासाठी एक मॉडेल निवडा  
4. मॉडेलसाठी एक सोपा प्रॉम्प्ट तयार करा  
5. आपल्या विनंतीला मॉडेल API कडे सादर करा!  


### 1. `azure-ai-inference` स्थापित करा


In [ ]:
%pip install azure-ai-inference

### 2. सहाय्यक लायब्ररी आयात करा आणि प्रमाणपत्र तयार करा


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. योग्य मॉडेल शोधणे  
GPT-4o आणि GPT-4o मिनी सारखे मॉडेल्स नैसर्गिक भाषा समजून घेऊ शकतात आणि तयार करू शकतात, आणि ते Microsoft Foundry Models कॅटलॉगमध्ये Meta, Mistral, Cohere आणि Microsoft या मॉडेल्सबरोबर उपलब्ध आहेत.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. प्रॉम्प्ट डिझाइन  

"मोठ्या भाषा मॉडेल्सचे जादू हे आहे की प्रचंड प्रमाणात मजकूरावर या भाकीत त्रुटी कमी करण्यासाठी प्रशिक्षण दिल्यामुळे, मॉडेल्स या भाकीतांसाठी उपयुक्त संकल्पना शिकतात. उदाहरणार्थ, ते अशा संकल्पना शिकतात"(1):

* कसे शब्दलेखन करावे
* व्याकरण कसे कार्य करते
* कसे परिभाषित करावे
* प्रश्नांची उत्तरे कशी द्यावीत
* संभाषण कसे चालवावे
* अनेक भाषांमध्ये लेखन कसे करावे
* कसे कोड करावे
* इत्यादी.

#### मोठ्या भाषा मॉडेलचे नियंत्रण कसे करावे  
"मोठ्या भाषा मॉडेलसाठी अनेक इनपुटपैकी सर्वात प्रभावी म्हणजे मजकूर प्रॉम्प्ट(1).

मोठ्या भाषा मॉडेल्स कसे प्रॉम्प्ट करून आउटपुट तयार करू शकतात:

सूचना: मॉडेलला काय हवे आहे ते सांगा
पूर्णता: तुम्हाला हवे असलेले प्रारंभ पूर्ण करण्यासाठी मॉडेलला प्रवृत्त करा
प्रदर्शन: तुम्हाला हवे असलेले मॉडेलला दाखवा, दोन्हीपैकी एकाने:
प्रॉम्प्टमध्ये काही उदाहरणे
फाइन-ट्यूनिंग प्रशिक्षण डेटासेटमध्ये अनेकशे किंवा हजारो उदाहरणे"



#### प्रॉम्प्ट तयार करण्यासाठी तीन मूलभूत मार्गदर्शक तत्त्वे आहेत:

**दाखवा आणि सांगा**. तुम्हाला काय हवे आहे हे सूचना, उदाहरणे किंवा दोन्हींच्या संयोजनाद्वारे स्पष्ट करा. जर तुम्हाला मॉडेलने आयटमची यादी अक्षरी क्रमाने मोजावी किंवा परिच्छेदाचा वर्गीकरण भावना संदर्भात करायची असेल तर ते काय हवे आहे हे दाखवा.

**चांगली गुणवत्ता डेटा प्रदान करा**. जर तुम्ही वर्गीकरणकर्ता तयार करत असाल किंवा मॉडेलने पॅटर्नचे पालन करावे असे इच्छित असाल तर पुरेशी उदाहरणे असावी याची खात्री करा. तुमची उदाहरणे योग्य आहेत का ते नक्की तपासा — मॉडेल सामान्यतः बेसिक स्पेलिंग चुका ओळखू शकते आणि उत्तर देऊ शकते, पण कधी कधी ते हे जानून की ही चूक जाणून केलेली आहे असे गृहित धरू शकते आणि हे उत्तरावर परिणाम करू शकते.

**तुमची सेटिंग्ज तपासा.** तापमान आणि टॉप_p सेटिंग्ज कशी निश्चित करतात की मॉडेल उत्तर तयार करण्यामध्ये कितपत निश्चित किंवा अनिश्चित आहे. जर तुम्हाला असा उत्तर हवा आहे ज्यात फक्त एकच योग्य उत्तर असेल, तर हे सेटिंग्ज कमी कराव्यात. जर तुम्हाला अधिक विविध उत्तर हवे असतील, तर सेटिंग्ज जास्त कराव्यात. लोकांच्या या सेटिंग्जबाबत केलेली सर्वात मोठी चूक म्हणजे ते यांना "चतुराई" किंवा "सर्जनशीलता" नियंत्रण समजतात.


स्रोत: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. सबमिट करा!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### एकाच कॉलची पुनरावृत्ती करा, परिणांम कसे आहेत?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## मजकूराचा सारांश करा  
#### आव्हान  
मजकूराच्या शेवटी 'tl;dr:' हा शब्द जोडून मजकूराचा सारांश करा. लक्षात घ्या की मॉडेल लगेचच अनेक कार्ये कशी पार पाडायची हे समजून घेते, अतिरिक्त सूचना न देता. तुम्ही tl;dr पेक्षा अधिक वर्णनात्मक सूचना वापरून मॉडेलच्या वर्तनात बदल करू शकता आणि तुम्हाला मिळणाऱ्या सारांशाचा सानुकूलन करू शकता(3).  

अलीकडील कामांनी अनेक NLP कार्ये आणि बेंचमार्कवर मोठे यश दाखवले आहे, जे मोठ्या प्रमाणावरील मजकूरावर प्री-ट्रेनिंग करून आणि नंतर विशिष्ट कार्यावर फाइन-ट्यूनिंग करून साध्य केले आहे. जरी सहसा task-agnostic आर्किटेक्चर वापरले जात असले तरी, या पद्धतीस हजारो किंवा दशलक्ष उदाहरणांच्या कार्य-विशिष्ट फाइन-ट्यूनिंग डेटासेटची गरज असते. त्याउलट, माणसं सामान्यतः काही उदाहरणं किंवा सोप्या सूचनांपासून नवीन भाषिक कार्य सहज पार पाडू शकतात - जे सध्याच्या NLP प्रणालींना अजूनही कठीण वाटते. येथे आम्ही दाखवतो की भाषा मॉडेल्सचे स्केल वाढवल्यावर कार्य-agnostic, कमी उदाहरणांवर आधारित कामगिरी मोठ्या प्रमाणात सुधारते, कधी कधी अगोदरच्या अत्याधुनिक फाइन-ट्यूनिंग पद्धतींशी स्पर्धा करू शकते.  



सारांश:  


# अनेक वापरांच्या बाबतीसाठी सराव  
1. मजकूर सारांशित करा  
2. मजकूर वर्गीकरण  
3. नवीन उत्पादनांची नावे तयार करा


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## मजकूर वर्गीकरण  
#### आव्हान  
वस्तूंना इन्फरन्स वेळेस दिलेल्या श्रेणीत वर्गीकृत करा. पुढील उदाहरणात, आम्ही प्रॉम्प्टमध्ये (*playground_reference) दोन्ही श्रेणी आणि वर्गीकरणासाठी मजकूर प्रदान करतो. 

ग्राहक चौकशी: नमस्कार, माझ्या लॅपटॉपच्या कीबोर्डवरील एक की नुकतीच तुटली आणि मला तिच्या बदल्यासाठी कळवावे लागेल:

वर्गीकृत श्रेणी:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## नवीन उत्पादन नावे तयार करा
#### आव्हान
उदाहरण शब्दांपासून उत्पादनाची नावे तयार करा. येथे आम्ही उत्पादनासाठी नाव तयार करताना संबंधित माहिती प्रदान करतो. तसेच आम्ही त्याच पॅटर्नचे उदाहरणही देतो जे आम्हाला हवं आहे. याद्वारे, आम्ही तापमानाचे मूल्य जास्त ठेवले आहे जेणेकरून अधिक अनियमित आणि नाविन्यपूर्ण प्रतिसाद मिळतील.

उत्पादनाचे वर्णन: घरगुती मिल्कशेक बनवणारा यंत्र
बीज शब्द: जलद, आरोग्यासाठी चांगले, कॉम्पॅक्ट.
उत्पादन नावे: HomeShaker, Fit Shaker, QuickShake, Shake Maker

उत्पादनाचे वर्णन: असा जोडा जो कोणत्याही पायाच्या आकाराला बसू शकतो.
बीज शब्द: जुळवून घेणारे, फिट, омनी-फिट.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# संदर्भ  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [टेक्स्ट वर्गीकृत करण्यासाठी GPT मॉडेल्सचे फाईन-ट्यूनिंगसाठी सर्वोत्तम पद्धती](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# अधिक मदतीसाठी  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# योगदानकर्ते
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
